<a href="https://colab.research.google.com/github/khouri-anes/plantdiseasedetection/blob/main/Copy_of_image_seg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import cv2
import os
from torch import nn
from google.colab import files
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


files.upload()
!unzip data_set.zip

def preprocess_image(image, train=True):

    img = np.array(image)

    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    l, a, b = cv2.split(img)

    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    image = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if train:

        transform = transforms.Compose([
            transforms.Resize((256,256)),
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]
            )
        ])

    else:

        transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]
            )
        ])

    image_tensor = transform(image)

    return image_tensor



In [ ]:
def collect_images(dataset_path):

    classes = sorted(os.listdir(dataset_path))
    class_to_idx = {c:i for i,c in enumerate(classes)}

    images = []

    valid_ext = (".jpg", ".jpeg", ".png")

    for c in classes:

        class_folder = os.path.join(dataset_path, c)

        for img in os.listdir(class_folder):

            if not img.lower().endswith(valid_ext):
                continue

            path = os.path.join(class_folder, img)

            images.append((path, class_to_idx[c]))

    return images, classes,class_to_idx

In [ ]:
class LeafDataset(Dataset):

    def __init__(self, data, train=True):

        self.data = data
        self.train = train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        img_path, label = self.data[idx]

        with Image.open(img_path) as img:
            image = img.convert("RGB")

        image = preprocess_image(image, train=self.train)

        return image, label

In [ ]:
dataset_path="data_set"

images, classes, class_to_idx = collect_images(dataset_path)

train_data, temp_data = train_test_split(
    images,
    test_size=0.3,
    stratify=[label for _,label in images],
    random_state=42
)
labels = [label for _, label in train_data]

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    stratify=[label for _,label in temp_data],
    random_state=42
)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [ ]:
train_dataset = LeafDataset(train_data, train=True)
val_dataset   = LeafDataset(val_data, train=False)
test_dataset  = LeafDataset(test_data, train=False)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class_num=len(classes)
model=models.resnet50(weights="DEFAULT",progress=True)
model.fc=nn.Linear(model.fc.in_features, class_num)
model=model.to(device)
loss_fun=nn.CrossEntropyLoss(weight=class_weights)
optimizer=torch.optim.Adam(model.parameters(),lr=0.0001)
epochs=20
patience = 5
best_val_loss = float("inf")
counter = 0


In [ ]:


for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_fun(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        loop.set_postfix(
            loss=running_loss/len(train_loader),
            acc=100*correct/total
        )

    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = loss_fun(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total

    print(f"\nValidation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), "best_model.pth")

        print("Validation improved, model saved!")

    else:

        counter += 1
        print(f"No improvement for {counter} epochs")

        if counter >= patience:
            print("Early stopping triggered!")
            break

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))

In [ ]:
from sklearn.metrics import classification_report
model.eval()

test_loss = 0
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    loop = tqdm(test_loader, desc="Testing")

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = loss_fun(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= len(test_loader)
test_acc = 100 * correct / total

print("\nTest Results")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

In [ ]:
target_layers = [model.layer4[-1]]

output_dir = "gradcam_results"
os.makedirs(output_dir, exist_ok=True)
cam = GradCAM(model=model, target_layers=target_layers)

labels_dir = "yolo_labels"
os.makedirs(labels_dir, exist_ok=True)


In [ ]:
for class_name in os.listdir(dataset_path):

    class_folder = os.path.join(dataset_path, class_name)

    for img_name in tqdm(os.listdir(class_folder), desc=class_name):

        img_path = os.path.join(class_folder, img_name)

        try:

            image = Image.open(img_path).convert("RGB")

            image_tensor = preprocess_image(image, train=False).unsqueeze(0).to(device)

            original_img = np.array(image)
            original_img = cv2.resize(original_img, (224, 224))
            rgb_img = original_img.astype(np.float32) / 255.0

            outputs = model(image_tensor)

            class_id = class_to_idx[class_name]

            targets = [ClassifierOutputTarget(class_id)]

            grayscale_cam = cam(
                input_tensor=image_tensor,
                targets=targets
            )[0]

            heatmap = (grayscale_cam * 255).astype("uint8")

            threshold = np.percentile(heatmap, 85)

            mask = (heatmap >= threshold).astype("uint8") * 255
            img_h, img_w = mask.shape

            contours, _ = cv2.findContours(
                mask,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )



            label_path = os.path.join(
                labels_dir,
                os.path.splitext(img_name)[0] + ".txt"
            )

            visualization = show_cam_on_image(
                rgb_img,
                grayscale_cam,
                use_rgb=True
            )
            plt.figure(figsize=(8,4))

            plt.subplot(1,2,1)
            plt.imshow(original_img)
            plt.title("Original Image")
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.imshow(visualization)
            plt.title("Grad-CAM Heatmap")
            plt.axis("off")

            plt.show()

            with open(label_path, "w") as f:

                for contour in contours:

                    if cv2.contourArea(contour) < 50:
                        continue

                    x, y, w, h = cv2.boundingRect(contour)

                    x_center = (x + w/2) / img_w
                    y_center = (y + h/2) / img_h
                    width = w / img_w
                    height = h / img_h

                    f.write(
                        f"{class_id} {x_center} {y_center} {width} {height}\n"
                    )

                    cv2.rectangle(
                        visualization,
                        (x, y),
                        (x + w, y + h),
                        (0, 255, 0),
                        2
                    )

            save_path = os.path.join(
                output_dir,
                f"{class_name}_{img_name}"
            )

            cv2.imwrite(
                save_path,
                cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR)
            )

        except Exception as e:
            print("Error:", img_name, e)

In [ ]:
!zip -r gradcam_results.zip gradcam_results
!zip -r yolo_labels.zip yolo_labels